<a href="https://colab.research.google.com/github/alottamyles/Hystrix/blob/2026fall_myles/Working_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import pandas as pd
import numpy as np
import io

# --- PART 1: GENERATE THE MOCK DATA CSV INSIDE COLAB ---
print("📦 Generating 'marketing_data.csv'...")

csv_data = """date,channel,spend,impressions,clicks,conversions
2026-10-01,Search,100.0,5126,342,39
2026-10-01,Social,100.0,9130,221,9
2026-10-01,Display,100.0,20734,103,3
2026-10-02,Search,100.0,5685,302,26
2026-10-02,Social,100.0,10391,187,6
2026-10-02,Display,100.0,20311,139,1
2026-10-03,Search,100.0,5184,307,36
2026-10-03,Social,100.0,10904,198,14
2026-10-03,Display,100.0,17047,139,4
2026-10-04,Search,100.0,5806,300,36
2026-10-04,Social,100.0,11638,213,13
2026-10-04,Display,100.0,18556,130,3
2026-10-05,Search,100.0,4020,322,31
2026-10-05,Social,100.0,9297,281,13
2026-10-05,Display,100.0,23433,88,2
2026-10-06,Search,100.0,4564,379,44
2026-10-06,Social,100.0,10139,260,11
2026-10-06,Display,100.0,15775,126,3
2026-10-07,Search,100.0,4205,330,28
2026-10-07,Social,100.0,9585,253,8
2026-10-07,Display,100.0,18073,85,2
2026-10-08,Search,100.0,5129,253,42
2026-10-08,Social,100.0,10777,193,6
2026-10-08,Display,100.0,19297,141,4
2026-10-09,Search,100.0,4269,344,40
2026-10-09,Social,100.0,11342,189,9
2026-10-09,Display,100.0,21863,132,4
2026-10-10,Search,100.0,5945,373,37
2026-10-10,Social,100.0,11112,164,5
2026-10-10,Display,100.0,23006,88,4
2026-10-11,Search,100.0,5152,385,35
2026-10-11,Social,100.0,9522,230,12
2026-10-11,Display,100.0,20892,120,4
2026-10-12,Search,100.0,4134,321,36
2026-10-12,Social,100.0,10977,182,11
2026-10-12,Display,100.0,17454,141,4
2026-10-13,Search,100.0,4804,348,36
2026-10-13,Social,100.0,9895,184,5
2026-10-13,Display,100.0,23110,82,1
2026-10-14,Search,100.0,5028,391,31
2026-10-14,Social,100.0,11044,204,8
2026-10-14,Display,100.0,21482,112,2"""

# Save locally to Colab's workspace
with open("marketing_data.csv", "w") as f:
    f.write(csv_data)
print("✅ CSV generated and saved successfully.\n")


# --- PART 2: THE AGENT LOGIC LOOP ---
def run_ad_optimization_agent(csv_path="marketing_data.csv"):
    df = pd.read_csv(csv_path)
    dates = sorted(df['date'].unique())
    channels = list(df['channel'].unique())

    # Baseline: Starting budget of $100 per channel
    current_budgets = {channel: 100.0 for channel in channels}
    floor_days_tracker = {channel: 0 for channel in channels}
    min_floor = 20.0  # Hard protective learning floor

    print("🚀 Starting Autonomous Agent Optimization Loop...")
    print("="*80)

    for day_idx, date in enumerate(dates):
        print(f"\n📅 DATE: {date} | Day {day_idx + 1}")
        day_data = df[df['date'] == date]

        # 1. Compute conversion performance
        cvr_scores = {}
        for channel in channels:
            ch_row = day_data[day_data['channel'] == channel]
            if not ch_row.empty:
                clicks = ch_row['clicks'].values[0]
                conversions = ch_row['conversions'].values[0]
                cvr_scores[channel] = conversions / clicks if clicks > 0 else 0
            else:
                cvr_scores[channel] = 0

        top_channel = max(cvr_scores, key=cvr_scores.get)
        worst_channel = min(cvr_scores, key=cvr_scores.get)

        # Print metrics for transparency
        print(f"   📊 CVR Performance -> Search: {cvr_scores['Search']:.2%} | Social: {cvr_scores['Social']:.2%} | Display: {cvr_scores['Display']:.2%}")

        # 2. Reasoning and budget shifts
        shift_percentage = 0.15  # Move 15% budget
        shift_amount = current_budgets[worst_channel] * shift_percentage

        new_worst_budget = max(current_budgets[worst_channel] - shift_amount, min_floor)
        actual_shifted_amount = current_budgets[worst_channel] - new_worst_budget
        new_top_budget = current_budgets[top_channel] + actual_shifted_amount

        # 3. Apply Guardrail Tracks
        if new_worst_budget <= min_floor:
            floor_days_tracker[worst_channel] += 1
        else:
            floor_days_tracker[worst_channel] = 0

        if floor_days_tracker[worst_channel] > 2:
            rationale_note = f"⚠️ GUARDRAIL TRIGGERED: {worst_channel} reached the hard floor for 2+ days. Giving $5 protection buffer to prevent total shutdown."
            new_worst_budget += 5.0
            new_top_budget -= 5.0
            floor_days_tracker[worst_channel] = 0
        else:
            rationale_note = f"🎯 OPTIMIZATION LOG: Shifting allocations dynamically to highest converting channel."

        # Update official states
        current_budgets[worst_channel] = round(new_worst_budget, 2)
        current_budgets[top_channel] = round(new_top_budget, 2)

        # 4. Print outputs and rationales
        print(f"   🤖 Action: Shifted allocation from {worst_channel} to {top_channel}")
        print(f"   📝 Rationale: {rationale_note} {top_channel} dominates with a {cvr_scores[top_channel]:.2%} conversion rate.")
        print(f"   💰 New Targets: Search: ${current_budgets['Search']} | Social: ${current_budgets['Social']} | Display: ${current_budgets['Display']}")
        print("-" * 80)

# Run the agent in Colab
run_ad_optimization_agent()


📦 Generating 'marketing_data.csv'...
✅ CSV generated and saved successfully.

🚀 Starting Autonomous Agent Optimization Loop...

📅 DATE: 2026-10-01 | Day 1
   📊 CVR Performance -> Search: 11.40% | Social: 4.07% | Display: 2.91%
   🤖 Action: Shifted allocation from Display to Search
   📝 Rationale: 🎯 OPTIMIZATION LOG: Shifting allocations dynamically to highest converting channel. Search dominates with a 11.40% conversion rate.
   💰 New Targets: Search: $115.0 | Social: $100.0 | Display: $85.0
--------------------------------------------------------------------------------

📅 DATE: 2026-10-02 | Day 2
   📊 CVR Performance -> Search: 8.61% | Social: 3.21% | Display: 0.72%
   🤖 Action: Shifted allocation from Display to Search
   📝 Rationale: 🎯 OPTIMIZATION LOG: Shifting allocations dynamically to highest converting channel. Search dominates with a 8.61% conversion rate.
   💰 New Targets: Search: $127.75 | Social: $100.0 | Display: $72.25
----------------------------------------------------